# 01 Load and QC Datalogger Data

This notebook loads ESP8266 datalogger measurement CSV files from **2026-05-29 onward only**. It preserves duplicated sensor columns from the two I2C buses by renaming repeated headers with `__bus0`, `__bus1`, ... suffixes.

In [ ]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

DATA_START_DATE = "20260529"
EXPECTED_INTERVAL_SECONDS = 20
GPS_STALE_THRESHOLD_MS = 120000
DATA_DIR = Path("..") / "test_data"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)
print("data directory", DATA_DIR.resolve())

In [ ]:
def data_file_date(path: Path) -> str:
    match = re.fullmatch(r"data_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def measurement_files(data_dir: Path = DATA_DIR, start_date: str = DATA_START_DATE) -> list[Path]:
    files = []
    for path in sorted(data_dir.glob("data_*.csv")):
        date = data_file_date(path)
        if date and date >= start_date:
            files.append(path)
    return files


def unique_headers(headers: list[str]) -> list[str]:
    total = Counter(headers)
    seen = defaultdict(int)
    out = []
    for header in headers:
        if total[header] == 1:
            out.append(header)
        else:
            bus = seen[header]
            out.append(f"{header}__bus{bus}")
            seen[header] += 1
    return out


def inspect_row_widths(path: Path) -> tuple[list[str], pd.DataFrame]:
    issues = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        raw_headers = next(reader)
        expected_width = len(raw_headers)
        for line_number, row in enumerate(reader, start=2):
            if len(row) != expected_width:
                issues.append({
                    "source_file": path.name,
                    "line_number": line_number,
                    "expected_columns": expected_width,
                    "actual_columns": len(row),
                })
    return raw_headers, pd.DataFrame(issues)


def read_measurement_file(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_headers, width_issues = inspect_row_widths(path)
    names = unique_headers(raw_headers)
    df = pd.read_csv(
        path,
        names=names,
        skiprows=1,
        na_values=["", "nan", "NaN", "NAN"],
        keep_default_na=True,
        on_bad_lines="skip",
    )
    df.insert(0, "source_file", path.name)
    df.insert(1, "source_date", data_file_date(path))
    return df, width_issues


files = measurement_files()
assert files, "No measurement files found from the configured start date."
print("Selected measurement files:")
for file in files:
    print(" -", file.name)

assert all(data_file_date(file) >= DATA_START_DATE for file in files)

In [ ]:
frames = []
width_issue_frames = []
for file in files:
    frame, issues = read_measurement_file(file)
    frames.append(frame)
    width_issue_frames.append(issues)

data = pd.concat(frames, ignore_index=True)
row_width_issues = pd.concat(width_issue_frames, ignore_index=True) if width_issue_frames else pd.DataFrame()

data["timestamp_utc"] = pd.to_datetime(data["timestamp_utc"], errors="coerce", utc=False)
if "timestamp_calc_utc" in data.columns:
    data["timestamp_calc_utc"] = pd.to_datetime(data["timestamp_calc_utc"], errors="coerce", utc=False)
else:
    data["timestamp_calc_utc"] = pd.NaT
if "gps_time_fresh" in data.columns:
    data["gps_time_fresh_bool"] = pd.to_numeric(data["gps_time_fresh"], errors="coerce").fillna(1).astype(bool)
else:
    data["gps_time_fresh_bool"] = True
gps_age = pd.to_numeric(data["gps_age_ms"], errors="coerce") if "gps_age_ms" in data.columns else pd.Series(pd.NA, index=data.index, dtype="Float64")
data["gps_age_too_high"] = gps_age > GPS_STALE_THRESHOLD_MS
data["gps_time_stale"] = (~data["gps_time_fresh_bool"]) | data["gps_age_too_high"].fillna(False)
if "timestamp_calc_source" in data.columns:
    calc_source = data["timestamp_calc_source"].astype("string").str.lower()
else:
    calc_source = pd.Series(pd.NA, index=data.index, dtype="string")
if "timestamp_boot_ms" in data.columns:
    boot_ms = pd.to_numeric(data["timestamp_boot_ms"], errors="coerce")
elif "uptime_ms" in data.columns:
    boot_ms = pd.to_numeric(data["uptime_ms"], errors="coerce")
else:
    boot_ms = pd.Series(pd.NA, index=data.index, dtype="Float64")
data["analysis_time"] = data["timestamp_utc"]
groups = data.groupby("boot_id", dropna=False) if "boot_id" in data.columns else [(None, data)]
for _, group in groups:
    fresh = (~group["gps_time_stale"]) & group["timestamp_utc"].notna() & boot_ms.loc[group.index].notna()
    if not fresh.any():
        continue
    base_idx = fresh[fresh].index[-1]
    base_utc = data.at[base_idx, "timestamp_utc"]
    base_boot = boot_ms.loc[base_idx]
    if pd.isna(base_boot):
        continue
    stale_idx = group.index[group["gps_time_stale"] & boot_ms.loc[group.index].notna()]
    data.loc[stale_idx, "analysis_time"] = base_utc + pd.to_timedelta(boot_ms.loc[stale_idx] - base_boot, unit="ms")
data["analysis_time"] = data["timestamp_calc_utc"].combine_first(data["analysis_time"])
data["time_corrected_from_uptime"] = (calc_source == "uptime").fillna(False) | (
    data["gps_time_stale"].fillna(False) & data["analysis_time"].notna() & data["timestamp_utc"].notna() & (data["analysis_time"] != data["timestamp_utc"])
)
data["file_date"] = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce").dt.date
data["date"] = data["analysis_time"].dt.date
data["hour"] = data["analysis_time"].dt.hour

print("Rows loaded:", len(data))
print("Columns loaded:", len(data.columns))
print("Timestamp parse failures:", int(data["timestamp_utc"].isna().sum()))
print("Rows with stale GPS time:", int(data["gps_time_stale"].sum()))
print("Rows corrected from uptime:", int(data["time_corrected_from_uptime"].sum()))
print("Row width issues:", len(row_width_issues))

data.head()

In [ ]:
def compute_gaps(df: pd.DataFrame, expected_interval_s: int = EXPECTED_INTERVAL_SECONDS) -> pd.DataFrame:
    pieces = []
    valid = df.dropna(subset=["analysis_time"]).sort_values(["source_file", "analysis_time"])
    for source_file, group in valid.groupby("source_file", sort=True):
        ts = group["analysis_time"].reset_index(drop=True)
        deltas = ts.diff().dt.total_seconds()
        gap = pd.DataFrame({
            "source_file": source_file,
            "previous_timestamp": ts.shift(1),
            "analysis_time": ts,
            "delta_seconds": deltas,
        })
        pieces.append(gap)
    out = pd.concat(pieces, ignore_index=True)
    out["missing_samples_est"] = np.maximum(
        0,
        np.floor(out["delta_seconds"].fillna(expected_interval_s) / expected_interval_s).astype(int) - 1,
    )
    return out


gaps_all = compute_gaps(data)
large_gaps = gaps_all[gaps_all["delta_seconds"] > 120].copy()
large_gaps["gap_minutes"] = large_gaps["delta_seconds"] / 60

duplicate_timestamps = (
    data.dropna(subset=["analysis_time"])
    .groupby(["source_file", "analysis_time"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

file_summary = (
    data.groupby("source_file", as_index=False)
    .agg(
        rows=("analysis_time", "size"),
        parseable_timestamps=("analysis_time", "count"),
        start_analysis_time=("analysis_time", "min"),
        end_analysis_time=("analysis_time", "max"),
        gps_sat_median=("nb_sat", "median"),
        hdop_median=("HDOP", "median"),
    )
)
gap_summary = large_gaps.groupby("source_file", as_index=False).agg(
    gaps_gt_120s=("delta_seconds", "size"),
    max_gap_s=("delta_seconds", "max"),
    missing_samples_est=("missing_samples_est", "sum"),
)
file_summary = file_summary.merge(gap_summary, on="source_file", how="left").fillna({
    "gaps_gt_120s": 0,
    "max_gap_s": 0,
    "missing_samples_est": 0,
})

file_summary

In [ ]:
print("Large gaps over 120 seconds:")
display(large_gaps[["source_file", "previous_timestamp", "analysis_time", "delta_seconds", "gap_minutes", "missing_samples_est"]])

print("Duplicate timestamps:", len(duplicate_timestamps))
display(duplicate_timestamps.head(20))

print("Row width issues:", len(row_width_issues))
display(row_width_issues.head(20))

In [ ]:
hourly_base = data.dropna(subset=["analysis_time"]).assign(
    date=lambda x: x["analysis_time"].dt.strftime("%Y-%m-%d"),
    hour=lambda x: x["analysis_time"].dt.hour,
)
hourly_coverage = hourly_base.groupby(["date", "hour"], as_index=False).agg(
    rows=("analysis_time", "size"),
    unique_timestamps=("analysis_time", "nunique"),
    stale_gps_rows=("gps_time_stale", "sum"),
    corrected_from_uptime_rows=("time_corrected_from_uptime", "sum"),
    max_gps_age_ms=("gps_age_ms", lambda x: pd.to_numeric(x, errors="coerce").max()),
)
hourly_coverage["duplicate_timestamp_rows"] = hourly_coverage["rows"] - hourly_coverage["unique_timestamps"]
hourly_coverage["expected_rows"] = 3600 / EXPECTED_INTERVAL_SECONDS
hourly_coverage["coverage_fraction"] = (hourly_coverage["unique_timestamps"] / hourly_coverage["expected_rows"]).clip(upper=1)
hourly_coverage["stale_gps_fraction"] = hourly_coverage["stale_gps_rows"] / hourly_coverage["rows"]
hourly_coverage["has_stale_gps"] = hourly_coverage["stale_gps_rows"] > 0

display(hourly_coverage.head(48))
display(hourly_coverage.query("date == '2026-05-31'"))

In [ ]:
status_files = sorted(DATA_DIR.glob("status_*.csv"))
if status_files:
    status_frames = []
    for path in status_files:
        frame = pd.read_csv(path, na_values=["", "nan", "NaN", "NAN"], keep_default_na=True)
        frame.insert(0, "source_file", path.name)
        status_frames.append(frame)
    status_data = pd.concat(status_frames, ignore_index=True)
    print("Status rows loaded:", len(status_data))
    display(status_data.head(50))
else:
    status_data = pd.DataFrame()
    print("No status_YYYYMMDD.csv files found yet; this is expected for pre-patch data.")

new_diag_cols = [
    "boot_id", "sample_counter", "uptime_ms", "timestamp_boot_ms", "reset_reason",
    "timestamp_calc_utc", "timestamp_calc_source",
    "gps_date_valid", "gps_time_fresh", "gps_location_valid", "gps_location_fresh",
    "gps_chars_processed", "gps_age_ms", "gps_location_age_ms", "gps_stale_count", "gps_recovery_count",
    "free_heap", "max_heap_block", "min_heap_block", "i2c_slow_count",
    "i2c_error_count", "i2c_recovery_count", "write_status",
]
present_diag_cols = [col for col in new_diag_cols if col in data.columns]
print("New firmware diagnostic columns present:", present_diag_cols)
if present_diag_cols:
    display(data[present_diag_cols].head())